# Week 6 — Multi-Head Attention + Positional Encoding + Encoder Block

**目标**：把 Week 5 的单头 attention 升级成多头，加位置编码和 FFN，组装成完整的 Transformer Encoder Block。

**产出**：
- `MultiHeadAttention`（带 padding mask）
- `SinusoidalPositionalEncoding` + `LearnablePositionalEncoding`
- `EncoderBlock`（Pre-LN 默认，Post-LN 可选）
- 在 Week 5 toy 任务上跑赢单头 + PE / Post-LN 两个消融实验


In [ ]:
# ── Bootstrap (Colab + local) ─────────────────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')


## 1. 多头的直觉

> 单头 attention 只能学一种"注意模式"。多头允许模型在不同子空间里并行地学多种模式。

具体做法：

1. 把 `d_model = 32` 切成 `num_heads = 4` 份，每头 `d_head = 8`。
2. 每头独立算 attention（并行的矩阵乘法）。
3. 拼回 `d_model`，再过一个输出投影 `W_o`。

直观上：有的头可能关注"相邻 token"，有的头关注"最大值 token"，有的头关注"句首"……多样性带来鲁棒性。

**实现关键**：不写 `for head in range(...)`。用 `reshape` + `transpose` 把 `(B, L, d)` 变成 `(B, h, L, d_head)` 一次算完。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """Multi-head self-attention with optional padding mask.

    Input  : x (B, L, d_model), optional padding_mask (B, L) with 1=keep, 0=pad
    Output : y (B, L, d_model), attn (B, h, L, L)
    """
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.h = num_heads
        self.d_head = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.last_attn = None

    def _split_heads(self, x):
        # (B, L, d) -> (B, h, L, d_head)
        B, L, _ = x.shape
        return x.view(B, L, self.h, self.d_head).transpose(1, 2)

    def _merge_heads(self, x):
        # (B, h, L, d_head) -> (B, L, d)
        B, h, L, dh = x.shape
        return x.transpose(1, 2).contiguous().view(B, L, h * dh)

    def forward(self, x, padding_mask=None):
        Q = self._split_heads(self.W_q(x))   # (B, h, L, d_head)
        K = self._split_heads(self.W_k(x))
        V = self._split_heads(self.W_v(x))

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)  # (B, h, L, L)

        if padding_mask is not None:
            # padding_mask: (B, L) -> (B, 1, 1, L)  broadcast across heads and queries
            m = padding_mask[:, None, None, :]
            scores = scores.masked_fill(m == 0, float('-inf'))

        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)              # (B, h, L, d_head)
        out = self._merge_heads(out)             # (B, L, d)
        self.last_attn = attn.detach()
        return self.W_o(out)


# shape sanity
mha = MultiHeadAttention(d_model=32, num_heads=4)
x = torch.randn(4, 8, 32)
y = mha(x)
print('output :', y.shape, '  attn :', mha.last_attn.shape)

# mask sanity: pad the last 3 positions of sample 0
pad = torch.ones(4, 8); pad[0, 5:] = 0
y2 = mha(x, padding_mask=pad)
print('masked attn[sample0, head0, query0] =', mha.last_attn[0, 0, 0].tolist())


## 2. Positional Encoding

Self-attention 对位置置换不变（permutation-equivariant）——如果你打乱输入 token 的顺序，输出也只是对应打乱。所以**必须显式注入位置信息**。

两种常见做法：

### 2.1 Sinusoidal（Vaswani et al.）

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

优点：不同频率叠加，模型能线性推导相对位置；能外推到训练时没见过的长度。

### 2.2 Learnable

一张 `(max_len, d_model)` 的 embedding 表，和 token embedding 一样被反向传播更新。

优点：简单，表达力强；缺点：无法外推到长度超出训练分布。


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()           # (L, 1)
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-(torch.log(torch.tensor(10000.0))) / d_model))  # (d/2,)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)   # (max_len, d_model), no grad

    def forward(self, x):
        L = x.size(1)
        return x + self.pe[:L].unsqueeze(0)


class LearnablePositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        L = x.size(1)
        pos = torch.arange(L, device=x.device)
        return x + self.pe(pos).unsqueeze(0)

# sanity
for cls in (SinusoidalPositionalEncoding, LearnablePositionalEncoding):
    pe = cls(d_model=32, max_len=64)
    z = pe(torch.zeros(1, 8, 32))
    print(cls.__name__, '->', z.shape)


### 2.3 可视化两种 PE

画成 `(L, d)` 热力图，肉眼对比。Sinusoidal 有清晰的周期条带；Learnable 初始化后是噪声，需要训练才会出现结构。


In [ ]:
import matplotlib.pyplot as plt

L, d = 64, 32
sin_pe = SinusoidalPositionalEncoding(d, L)
lrn_pe = LearnablePositionalEncoding(d, L)
sin_mat = sin_pe.pe[:L].numpy()
lrn_mat = lrn_pe.pe(torch.arange(L)).detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, mat, title in zip(axes, [sin_mat, lrn_mat],
                          ['Sinusoidal PE (fixed)', 'Learnable PE (random init)']):
    im = ax.imshow(mat, aspect='auto', cmap='RdBu_r', vmin=-1.2, vmax=1.2)
    ax.set_xlabel('embedding dim'); ax.set_ylabel('position')
    ax.set_title(f'{title}  shape={mat.shape}')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


## 3. Encoder Block（Pre-LN）

一个标准的 Transformer encoder block 做两件事：

1. multi-head self-attention + 残差
2. position-wise FFN（`Linear(d → 4d) → GELU → Linear(4d → d)`）+ 残差

**Pre-LN 写法**（本 notebook 默认，训练更稳）：
```
y = x + MHA(LayerNorm(x))
z = y + FFN(LayerNorm(y))
```

**Post-LN 写法**（原论文版本）：
```
y = LayerNorm(x + MHA(x))
z = LayerNorm(y + FFN(y))
```

直观差异：Pre-LN 始终给 MHA/FFN 的输入是"归一化过"的，稳定；Post-LN 把归一化放在残差之后，深层堆叠或高学习率时容易数值爆炸。


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, mult: int = 4, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, mult * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mult * d_model, d_model),
        )
    def forward(self, x): return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_mult: int = 4,
                 dropout: float = 0.0, pre_ln: bool = True):
        super().__init__()
        self.pre_ln = pre_ln
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, ff_mult, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None):
        if self.pre_ln:
            x = x + self.drop(self.mha(self.ln1(x), padding_mask))
            x = x + self.drop(self.ffn(self.ln2(x)))
        else:
            x = self.ln1(x + self.drop(self.mha(x, padding_mask)))
            x = self.ln2(x + self.drop(self.ffn(x)))
        return x

# shape + backward sanity
blk = EncoderBlock(d_model=32, num_heads=4)
x = torch.randn(4, 8, 32, requires_grad=True)
y = blk(x)
print('forward :', y.shape)
y.sum().backward()
print('backward ok, grad norm =', x.grad.norm().item())
print('params  :', sum(p.numel() for p in blk.parameters()))


## 4. 完整小模型：`TransformerClassifier`

结构：`Embedding → +PE → N × EncoderBlock → mean-pool → Linear`

在 Week 5 的 toy 任务上跑，跟单头 attention 对比。


In [ ]:
VOCAB, SEQ_LEN, THRESH = 16, 8, 50

def make_dataset(n):
    x = torch.randint(0, VOCAB, (n, SEQ_LEN))
    y = (x.cumsum(dim=1).max(dim=1).values > THRESH).long()
    return x, y

torch.manual_seed(42)
X_tr, y_tr = make_dataset(4096)
X_va, y_va = make_dataset(1024)
print('train', X_tr.shape, 'pos rate', y_tr.float().mean().item())

class TransformerClassifier(nn.Module):
    def __init__(self, vocab=VOCAB, d=32, num_heads=4, n_layers=2,
                 seq_len=SEQ_LEN, pe_kind='sinusoidal', use_pe=True, pre_ln=True):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.use_pe = use_pe
        if use_pe:
            self.pe = (SinusoidalPositionalEncoding(d, seq_len)
                       if pe_kind == 'sinusoidal'
                       else LearnablePositionalEncoding(d, seq_len))
        self.blocks = nn.ModuleList([
            EncoderBlock(d, num_heads, pre_ln=pre_ln) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, 1)

    def forward(self, x):
        h = self.emb(x)
        if self.use_pe:
            h = self.pe(h)
        for blk in self.blocks:
            h = blk(h)
        h = self.ln_f(h).mean(dim=1)
        return self.head(h).squeeze(-1)


In [ ]:
def train_clf(model, X_tr, y_tr, X_va, y_va, steps=2000, bs=128, lr=1e-3,
              clip=1.0):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses, diverged = [], False
    X_tr_d, y_tr_d = X_tr.to(device), y_tr.float().to(device)
    X_va_d, y_va_d = X_va.to(device), y_va.float().to(device)
    for step in range(steps):
        idx = torch.randint(0, len(X_tr_d), (bs,), device=device)
        logit = model(X_tr_d[idx])
        loss = F.binary_cross_entropy_with_logits(logit, y_tr_d[idx])
        opt.zero_grad(); loss.backward()
        if clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        losses.append(loss.item())
        if not torch.isfinite(loss):
            diverged = True; break
    with torch.no_grad():
        acc = ((model(X_va_d) > 0) == (y_va_d > 0.5)).float().mean().item()
    return losses, acc, diverged


torch.manual_seed(42)
trf = TransformerClassifier(n_layers=2)
n_params = sum(p.numel() for p in trf.parameters())
print('TransformerClassifier params:', n_params)
trf_losses, trf_acc, _ = train_clf(trf, X_tr, y_tr, X_va, y_va, steps=2000, lr=1e-3)
print(f'2-layer Transformer valid acc = {trf_acc:.3f}')


### 4.1 与 Week 5 单头模型对比

重建一个单头 attention baseline（同样 `d=32`，没有 PE、没有 FFN、1 层）跑一下看 loss 曲线。


In [ ]:
class SingleHeadBaseline(nn.Module):
    def __init__(self, vocab=VOCAB, d=32):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.W_q = nn.Linear(d, d, bias=False)
        self.W_k = nn.Linear(d, d, bias=False)
        self.W_v = nn.Linear(d, d, bias=False)
        self.W_o = nn.Linear(d, d, bias=False)
        self.head = nn.Linear(d, 1)
        self.d = d
    def forward(self, x):
        h = self.emb(x)
        Q, K, V = self.W_q(h), self.W_k(h), self.W_v(h)
        scores = Q @ K.transpose(-2, -1) / (self.d ** 0.5)
        attn = scores.softmax(-1)
        h = self.W_o(attn @ V).mean(1)
        return self.head(h).squeeze(-1)

torch.manual_seed(42)
single = SingleHeadBaseline()
single_losses, single_acc, _ = train_clf(single, X_tr, y_tr, X_va, y_va, steps=2000, lr=1e-3)
print(f'single-head acc = {single_acc:.3f}   transformer acc = {trf_acc:.3f}')

import numpy as np
def smooth(xs, k=50):
    xs = np.asarray(xs)
    if len(xs) < k: return xs
    return np.convolve(xs, np.ones(k)/k, mode='valid')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(smooth(single_losses), label=f'SingleHead  acc={single_acc:.3f}')
ax.plot(smooth(trf_losses),    label=f'2L Transformer (MHA+PE+FFN)  acc={trf_acc:.3f}')
ax.set_xlabel('step (smoothed)'); ax.set_ylabel('loss')
ax.set_title('Week 5 single-head vs Week 6 2-layer Transformer')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 5. 形状与反向传播验证

契约检查：输入 `(B=4, L=8, d=32)` → 输出 `(4, 8, 32)`；梯度能顺畅回传。顺便打印参数数量，让你对规模有感觉。


In [ ]:
blk = EncoderBlock(d_model=32, num_heads=4)
x = torch.randn(4, 8, 32, requires_grad=True)
y = blk(x)
assert y.shape == (4, 8, 32), f'unexpected output shape {y.shape}'
y.sum().backward()
assert x.grad is not None and torch.isfinite(x.grad).all(), 'bad gradient'
print('EncoderBlock shape OK :', tuple(y.shape))
print('EncoderBlock params   :', sum(p.numel() for p in blk.parameters()))
print('TransformerClassifier params :', sum(p.numel() for p in TransformerClassifier(n_layers=2).parameters()))


## 6. 消融一：去掉 Positional Encoding

理论预期：没有 PE，self-attention 看不见顺序，而"前缀和"这个 toy 任务高度依赖顺序——loss 应当几乎学不下来。


In [ ]:
torch.manual_seed(42)
no_pe = TransformerClassifier(n_layers=2, use_pe=False)
no_pe_losses, no_pe_acc, _ = train_clf(no_pe, X_tr, y_tr, X_va, y_va, steps=2000, lr=1e-3)
print(f'no-PE acc = {no_pe_acc:.3f}   with-PE acc = {trf_acc:.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(smooth(trf_losses),   label=f'with PE (sin)  acc={trf_acc:.3f}')
ax.plot(smooth(no_pe_losses), label=f'no PE          acc={no_pe_acc:.3f}')
ax.set_xlabel('step (smoothed)'); ax.set_ylabel('loss')
ax.set_title('Ablation: with / without Positional Encoding')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 7. 消融二：Pre-LN vs Post-LN（高学习率下）

在 `lr=1e-3` 下两者都能跑。把 lr 拉高到 `5e-3`，Post-LN 通常会表现出明显的不稳定（初期 loss 震荡、甚至 NaN）。Pre-LN 则相对平稳。


In [ ]:
HIGH_LR = 5e-3

torch.manual_seed(0)
pre = TransformerClassifier(n_layers=2, pre_ln=True)
pre_losses, pre_acc, pre_div = train_clf(pre, X_tr, y_tr, X_va, y_va,
                                         steps=2000, lr=HIGH_LR)
print(f'Pre-LN  @ lr={HIGH_LR}  acc={pre_acc:.3f}  diverged={pre_div}')

torch.manual_seed(0)
post = TransformerClassifier(n_layers=2, pre_ln=False)
post_losses, post_acc, post_div = train_clf(post, X_tr, y_tr, X_va, y_va,
                                            steps=2000, lr=HIGH_LR)
print(f'Post-LN @ lr={HIGH_LR}  acc={post_acc:.3f}  diverged={post_div}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(smooth(pre_losses),  label=f'Pre-LN  acc={pre_acc:.3f}')
ax.plot(smooth(post_losses), label=f'Post-LN acc={post_acc:.3f}')
ax.set_xlabel('step (smoothed)'); ax.set_ylabel('loss')
ax.set_title(f'Pre-LN vs Post-LN at lr={HIGH_LR}')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 8. 复盘三问（内联答案）

**Q1. 为什么需要 Positional Encoding？**

Self-attention 对位置置换不变：把输入 token 打乱顺序后，输出只是对应打乱，模型并不"知道"哪个 token 是第一个。对于依赖顺序的任务（如前缀和、语言建模），这是致命的——消融曲线里可以看到去掉 PE 后 loss 几乎不下降。

Sinusoidal PE 用不同频率的正余弦叠加，让模型能以线性组合表达"位置 p 到位置 p+k 的相对位移"；Learnable PE 简单直接但无法外推到训练时没见过的长度。

**Q2. Pre-LN 和 Post-LN？**

Post-LN（原论文）：`LN(x + Sublayer(x))`，LayerNorm 放残差之后。深堆叠或高学习率时，残差分支的方差会被 LN 反复收缩，梯度信号衰减，训练不稳定，通常需要 warmup 很久才能启动。

Pre-LN（现代实现默认）：`x + Sublayer(LN(x))`，LayerNorm 放残差之前。每个 sublayer 拿到的输入都是标准化过的，残差路径保持恒等，梯度能无损回传。可以用更大 lr，更少 warmup，深度堆叠更鲁棒。

上面的消融实验里，`lr=5e-3` 下 Post-LN 明显比 Pre-LN 震荡更剧烈——这是 Pre-LN 被现代实现（GPT-2/3、LLaMA、PaLM）普遍采用的原因。

**Q3. 多头为什么比单头好？**

单头只能学一种注意模式（一个权重矩阵定义一个"相似度"）。多头把 `d_model` 切成多个子空间并行算 attention，每个头自由地关注不同的结构（相邻 token、极值 token、句首、远距离依赖……），然后拼接融合。直觉上和 CNN 多通道是同一种"多视图特征"思路。

---

下周（Week 7）：把这套 Encoder Block 堆到 4 层，加 `[CLS]` token / Noam warmup，产出 MVP v0.3 的第一个 Transformer。
